# 🏠 Домашнее задание 17. SQL: SELECT, WHERE, ORDER BY, LIMIT, DISTINCT

**Как сдавать:**
1. **Файл → Сохранить копию на Диске.**
2. Запустите ячейку «⚙️ Подготовка базы» и загрузите `orders.csv`.
3. Запустите ячейку «✅ Проверка».
4. Решите задачи трёх блоков. После каждого решения вызывайте `check(номер, результат)`.
   - **Блок А (1–3):** дан pandas, пишете SQL.
   - **Блок Б (4–6):** дан SQL, пишете pandas.
   - **Блок В (7–10):** дано условие, пишете на обоих языках.
5. Когда всё ✅, отправьте ссылку на ноутбук преподавателю.

**Что есть в базе `shop.db`:** одна таблица **`orders`** (в pandas это `df`):
`order_id`, `date`, `customer_id`, `city`, `category`, `product`, `price`, `qty`, `status`.

> 💡 `check()` не смотрит на названия столбцов и индекс, но **порядок столбцов и строк важен**: берите столбцы ровно в том порядке, как в условии. В переводах результат должен совпасть с результатом данного кода.

In [178]:
# @title ⚙️ Подготовка базы shop.db (просто запусти эту ячейку)
import os, sqlite3
import pandas as pd

def upload_csv():
    """Загружает orders.csv (и по желанию orders_raw.csv) в Colab."""
    if os.path.exists('orders.csv'):
        print('orders.csv уже загружен ✅')
        return
    from google.colab import files
    print('Выбери на компьютере orders.csv (можно сразу вместе с orders_raw.csv) 👇')
    for name, data in files.upload().items():
        target = 'orders_raw.csv' if 'raw' in name else 'orders.csv'
        with open(target, 'wb') as f:
            f.write(data)
        print(f'{name} → {target}')

upload_csv()
conn = sqlite3.connect('shop.db')   # подключаемся к базе (файла нет — SQLite его создаст)
df = pd.read_csv('orders.csv')      # читаем CSV в pandas

df.to_sql('orders', conn, if_exists='replace', index=False)   # ← вся база одной строкой!
def sql(query):
    """Выполняет SQL-запрос и возвращает результат как таблицу pandas."""
    return pd.read_sql_query(query, conn)
print(f'База готова ✅  Таблица orders: {sql("SELECT COUNT(*) AS n FROM orders").n[0]} строк')
print('Для pandas та же таблица лежит в переменной df')

orders.csv уже загружен ✅
База готова ✅  Таблица orders: 500 строк
Для pandas та же таблица лежит в переменной df


In [179]:
# @title ✅ Проверка (запусти, ничего не меняй)
import hashlib

def _norm(v):
    if isinstance(v, str):
        return v.strip()
    if v is None or pd.isna(v):
        return 'NULL'
    f = float(v)
    return str(int(f)) if f.is_integer() else f'{f:.2f}'

def _fingerprint(res, ordered):
    if isinstance(res, pd.Series):
        res = res.to_frame()
    elif not isinstance(res, pd.DataFrame):
        res = pd.DataFrame({'value': list(res)})
    rows = ['|'.join(_norm(v) for v in row) for row in res.itertuples(index=False)]
    if not ordered:
        rows = sorted(rows)
    return hashlib.md5('\n'.join(rows).encode('utf-8')).hexdigest(), res.shape

ANSWERS = {
    1: ('aca3188a94de669704734511d9b8b35e', [4, 2], True),
    2: ('b6c14504b8b59727ca4a00be1dbc648f', [38, 4], True),
    3: ('7c68d0e3ab40a8eb11c35568fdcc9875', [9, 4], True),
    4: ('4c1ab23ba240cad43b16dce58fccffd8', [14, 3], True),
    5: ('09f9aff6e61f7c0f590843cd09ebddfa', [10, 4], True),
    6: ('f8a6720b23f22d96ef53581edcf576aa', [21, 3], True),
    7: ('b88fe9663ea6a182d8ebe42b2fd5d8dc', [2, 1], True),
    8: ('9f8972797080b1f517cb9dd8e7a85ad8', [10, 5], True),
    9: ('494b2fd27a1006e10f29a593b62aad92', [35, 1], True),
    10: ('01828a1e09b0f2329433d6da03f7e305', [5, 4], True),
}

def check(n, result):
    """Проверяет ответ к задаче n. Названия столбцов и индекс не важны, важны значения, их порядок и порядок строк."""
    h, (r, c), ordered = ANSWERS[n]
    got_h, (got_r, got_c) = _fingerprint(result, ordered)
    if got_h == h:
        print(f'✅ Задача {n}: верно! (строк: {got_r}, столбцов: {got_c})')
    elif (got_r, got_c) != (r, c):
        print(f'❌ Задача {n}: ожидается таблица {r}×{c} (строки×столбцы), а у тебя {got_r}×{got_c}')
    else:
        print(f'❌ Задача {n}: размер верный ({r} × {c}), но значения или порядок строк/столбцов отличаются')

---
## Блок А: pandas → SQL (задачи 1–3)
Дан код на pandas. Сначала разберитесь, что он делает, потом напишите **тот же запрос на SQL**.

### Задача 1: переведи на SQL
```python
(df.loc[df['price'] < 50000, ['product', 'price']]
   .drop_duplicates()
   .sort_values('price'))
```

In [180]:
# 🟦 SQL
res = sql("""
SELECT DISTINCT "product", "price"
from orders
WHERE "price" < 50000
ORDER BY "price"
""")
check(1, res)
res

✅ Задача 1: верно! (строк: 4, столбцов: 2)


,product,price
0,Мышь Logitech M185,7500
1,Клавиатура Logitech K120,12000
2,Power bank Xiaomi 20000,14000
3,Наушники JBL Tune 520,25000


### Задача 2: переведи на SQL
```python
(df.loc[(df['status'] == 'оплачен') & (df['qty'] == 3), ['order_id', 'date', 'product', 'qty']]
   .sort_values('order_id'))
```

In [181]:
# 🟦 SQL
res = sql("""
SELECT DISTINCT "order_id", "date", "product", "qty"
FROM orders
WHERE "status" == 'оплачен' AND "qty" == 3
ORDER BY "order_id"
""")
check(2, res)
res

✅ Задача 2: верно! (строк: 38, столбцов: 4)


,order_id,date,product,qty
0,ORD-0017,2026-03-09,Наушники JBL Tune 520,3
1,ORD-0019,2026-03-09,Наушники JBL Tune 520,3
2,ORD-0022,2026-03-11,Клавиатура Logitech K120,3
3,ORD-0049,2026-03-24,Клавиатура Logitech K120,3
4,ORD-0055,2026-04-02,Мышь Logitech M185,3
5,ORD-0056,2026-04-02,Power bank Xiaomi 20000,3
6,ORD-0061,2026-04-06,Клавиатура Logitech K120,3
7,ORD-0072,2026-04-13,Наушники JBL Tune 520,3
8,ORD-0086,2026-04-20,Наушники JBL Tune 520,3
9,ORD-0097,2026-04-28,Power bank Xiaomi 20000,3


### Задача 3: переведи на SQL
```python
(df.loc[df['product'].isin(['iPhone 15', 'MacBook Air M3']) &
        df['date'].between('2026-09-01', '2026-09-30'),
        ['order_id', 'date', 'product', 'status']]
   .sort_values(['date', 'order_id']))
```

In [182]:
# 🟦 SQL я только сейчас вспомнил, что двойные кавычки не нужны
res = sql("""
SELECT order_id, date, product, status
FROM orders
WHERE product IN ('iPhone 15', 'MacBook Air M3')
AND date BETWEEN ('2026-09-01') AND ('2026-09-30')
ORDER BY date, order_id
""")
check(3, res)
res

✅ Задача 3: верно! (строк: 9, столбцов: 4)


,order_id,date,product,status
0,ORD-0366,2026-09-01,iPhone 15,оплачен
1,ORD-0383,2026-09-07,MacBook Air M3,отменён
2,ORD-0386,2026-09-08,iPhone 15,оплачен
3,ORD-0400,2026-09-12,MacBook Air M3,отменён
4,ORD-0410,2026-09-13,iPhone 15,оплачен
5,ORD-0413,2026-09-18,MacBook Air M3,оплачен
6,ORD-0423,2026-09-21,MacBook Air M3,оплачен
7,ORD-0425,2026-09-22,MacBook Air M3,оплачен
8,ORD-0426,2026-09-22,iPhone 15,оплачен


---
## Блок Б: SQL → pandas (задачи 4–6)
Дан SQL-запрос. Напишите **то же самое на pandas**.

### Задача 4: переведи на pandas
```sql
SELECT order_id, date, product
FROM orders
WHERE city = 'Караганда'
  AND date BETWEEN '2026-10-01' AND '2026-10-31'
ORDER BY date, order_id
```

In [183]:
# 🐼 pandas
res_pd = (df.loc[(df["city"] == "Караганда") & (df["date"].between("2026-10-01", "2026-10-31")),
                 ["order_id", "date", "product"]]).sort_values(["date", "order_id"])
...
check(4, res_pd)
res_pd

✅ Задача 4: верно! (строк: 14, столбцов: 3)


,order_id,date,product
445,ORD-0446,2026-10-02,Монитор LG 27 дюймов
447,ORD-0448,2026-10-03,Lenovo IdeaPad 5
451,ORD-0452,2026-10-04,Power bank Xiaomi 20000
453,ORD-0454,2026-10-04,Samsung Galaxy Tab A9
457,ORD-0458,2026-10-06,Мышь Logitech M185
459,ORD-0460,2026-10-08,Клавиатура Logitech K120
460,ORD-0461,2026-10-08,Samsung Galaxy Tab A9
464,ORD-0465,2026-10-10,Мышь Logitech M185
465,ORD-0466,2026-10-10,MacBook Air M3
467,ORD-0468,2026-10-11,Power bank Xiaomi 20000


### Задача 5: переведи на pandas
```sql
SELECT order_id, date, city, product
FROM orders
ORDER BY date DESC, order_id DESC
LIMIT 10
```

In [184]:
# 🐼 pandas
res_pd = (df[["order_id", "date", "city", "product"]].sort_values(by=["date", "order_id"], ascending=[False, False]).head(10))
...
check(5, res_pd)
res_pd

✅ Задача 5: верно! (строк: 10, столбцов: 4)


,order_id,date,city,product
499,ORD-0500,2026-10-31,Астана,Монитор LG 27 дюймов
498,ORD-0499,2026-10-31,Алматы,ASUS VivoBook 15
497,ORD-0498,2026-10-30,Астана,ASUS VivoBook 15
496,ORD-0497,2026-10-30,Шымкент,Клавиатура Logitech K120
495,ORD-0496,2026-10-29,Алматы,Xiaomi Redmi Note 13
494,ORD-0495,2026-10-28,Актобе,Samsung Galaxy Tab A9
493,ORD-0494,2026-10-28,Астана,Наушники JBL Tune 520
492,ORD-0493,2026-10-28,Караганда,iPad 10
491,ORD-0492,2026-10-27,Актобе,Power bank Xiaomi 20000
490,ORD-0491,2026-10-27,Алматы,iPhone 15


### Задача 6: переведи на pandas
```sql
SELECT order_id, product, qty
FROM orders
WHERE product LIKE '%Logitech%' AND city = 'Шымкент'
ORDER BY order_id
```

In [185]:
# 🐼 pandas
res_pd = df.loc[(df["product"].str.contains("Logitech")) & (df["city"] == "Шымкент"), ["order_id", "product", "qty"]].sort_values("order_id")
...
check(6, res_pd)
res_pd

✅ Задача 6: верно! (строк: 21, столбцов: 3)


,order_id,product,qty
10,ORD-0011,Клавиатура Logitech K120,1
32,ORD-0033,Клавиатура Logitech K120,1
50,ORD-0051,Клавиатура Logitech K120,2
101,ORD-0102,Мышь Logitech M185,1
117,ORD-0118,Мышь Logitech M185,1
132,ORD-0133,Мышь Logitech M185,2
136,ORD-0137,Мышь Logitech M185,2
145,ORD-0146,Мышь Logitech M185,3
146,ORD-0147,Клавиатура Logitech K120,3
159,ORD-0160,Мышь Logitech M185,2


---
## Блок В: с нуля (задачи 7–10)
Дано только условие. Решите **на SQL и на pandas**.

### Задача 7
Какие **категории** товаров покупали клиенты из **Атырау**? Только столбец `category`, **без повторов**, по алфавиту.

In [186]:
# 🟦 SQL
res = sql("""
SELECT DISTINCT category
FROM orders
WHERE city = 'Атырау'
ORDER BY category
""")
check(7, res)
res

✅ Задача 7: верно! (строк: 2, столбцов: 1)


,category
0,Аксессуары
1,Мониторы


In [187]:
# 🐼 pandas
res_pd = (df.loc[(df["city"] == "Атырау"), ["category"]].sort_values("category"))
...
check(7, res_pd)
res_pd

✅ Задача 7: верно! (строк: 2, столбцов: 1)


,category
147,Аксессуары
463,Мониторы


### Задача 8
**Оплаченные** заказы с суммой `price * qty` **от 200 000 до 500 000 ₸** включительно. Столбцы `order_id`, `product`, `price`, `qty`, `total`. Сортировка: `total` по убыванию, при равенстве `order_id` по возрастанию. Только **первые 10**.

In [188]:
# 🟦 SQL
res = sql("""
SELECT order_id, product, price, qty, price * qty AS total
FROM orders
WHERE status = 'оплачен'
AND total BETWEEN 200000 AND 500000
ORDER BY total DESC, order_id ASC
LIMIT 10
""")
check(8, res)
res

✅ Задача 8: верно! (строк: 10, столбцов: 5)


,order_id,product,price,qty,total
0,ORD-0050,iPad 10,239000,2,478000
1,ORD-0155,iPad 10,239000,2,478000
2,ORD-0020,iPhone 15,449000,1,449000
3,ORD-0023,iPhone 15,449000,1,449000
4,ORD-0024,iPhone 15,449000,1,449000
5,ORD-0067,iPhone 15,449000,1,449000
6,ORD-0080,iPhone 15,449000,1,449000
7,ORD-0098,iPhone 15,449000,1,449000
8,ORD-0111,iPhone 15,449000,1,449000
9,ORD-0117,iPhone 15,449000,1,449000


In [189]:
# 🐼 pandas здесь уже без ИИ тяжело, не могу пока вникнуть
df8 = df.copy()
df8["total"] = df8["price"] * df8["qty"]
res_pd = (
    df8.loc[(df8["status"] == "оплачен") & df8["total"].between(200000, 500000)]
    .sort_values(by=["total", "order_id"], ascending=[False, True])
    [["order_id", "product", "price", "qty", "total"]]
    .head(10)
)
...
check(8, res_pd)
res_pd

✅ Задача 8: верно! (строк: 10, столбцов: 5)


,order_id,product,price,qty,total
49,ORD-0050,iPad 10,239000,2,478000
154,ORD-0155,iPad 10,239000,2,478000
19,ORD-0020,iPhone 15,449000,1,449000
22,ORD-0023,iPhone 15,449000,1,449000
23,ORD-0024,iPhone 15,449000,1,449000
66,ORD-0067,iPhone 15,449000,1,449000
79,ORD-0080,iPhone 15,449000,1,449000
97,ORD-0098,iPhone 15,449000,1,449000
110,ORD-0111,iPhone 15,449000,1,449000
116,ORD-0117,iPhone 15,449000,1,449000


### Задача 9
Какие клиенты хотя бы раз **отменяли** заказ? Только столбец `customer_id`, **без повторов**, по возрастанию.

In [190]:
#res = sql("""
#SELECT DISTINCT status
#FROM orders
#""")
#res

In [191]:
# 🟦 SQL
res = sql("""
SELECT DISTINCT customer_id
FROM orders
WHERE status = 'отменён'
ORDER BY customer_id
""")
check(9, res)
res

✅ Задача 9: верно! (строк: 35, столбцов: 1)


,customer_id
0,C001
1,C006
2,C008
3,C011
4,C013
5,C019
6,C020
7,C028
8,C035
9,C037


In [192]:
print(df["status"].unique())
print(df.shape)
print(df["status"].value_counts(dropna=False))


<StringArray>
['оплачен', 'отменён']
Length: 2, dtype: str
(500, 9)
status
оплачен    455
отменён     45
Name: count, dtype: int64


In [193]:
# 🐼 pandas
res_pd = df.loc[(df["status"] == "отменён"), ["customer_id"]].drop_duplicates().sort_values("customer_id")

# Пожалуйста, дайте фидбэк, я так и не смог понять, почему ошибка
...
check(9, res_pd)
res_pd

✅ Задача 9: верно! (строк: 35, столбцов: 1)


,customer_id
469,C001
382,C006
108,C008
320,C011
69,C013
67,C019
225,C020
399,C028
32,C035
153,C037


### Задача 10
**Оплаченные** заказы **смартфонов** из всех городов, **кроме Алматы и Астаны** (`NOT IN`). Сортировка по `date`, затем по `order_id`. Выведите **вторую пятёрку** (заказы с 6-го по 10-й). Столбцы `order_id`, `date`, `city`, `product`.

In [194]:
# 🟦 SQL
res = sql("""
SELECT order_id, date, city, product
FROM orders
WHERE category = "Смартфоны" AND status = "оплачен" AND city NOT IN ("Алматы", "Астана")
ORDER BY date ASC, order_id ASC
LIMIT 5 OFFSET 5
""")
check(10, res)
res

✅ Задача 10: верно! (строк: 5, столбцов: 4)


,order_id,date,city,product
0,ORD-0067,2026-04-11,Шымкент,iPhone 15
1,ORD-0089,2026-04-22,Караганда,Xiaomi Redmi Note 13
2,ORD-0093,2026-04-25,Павлодар,Samsung Galaxy A55
3,ORD-0099,2026-04-29,Шымкент,Samsung Galaxy A55
4,ORD-0113,2026-05-08,Караганда,Samsung Galaxy A55


In [195]:
# 🐼 pandas
res_pd = (df.loc[(df["category"] == "Смартфоны")
                & (df["status"] == "оплачен")
                & (~df["city"].isin(["Алматы", "Астана"])),
                ["order_id", "date", "city", "product"]].sort_values(["date", "order_id"]).iloc[5:10].reset_index(drop=True))
# Тоже самую ошибку видимо допускаю, не могу понять
...
check(10, res_pd)
res_pd

✅ Задача 10: верно! (строк: 5, столбцов: 4)


,order_id,date,city,product
0,ORD-0067,2026-04-11,Шымкент,iPhone 15
1,ORD-0089,2026-04-22,Караганда,Xiaomi Redmi Note 13
2,ORD-0093,2026-04-25,Павлодар,Samsung Galaxy A55
3,ORD-0099,2026-04-29,Шымкент,Samsung Galaxy A55
4,ORD-0113,2026-05-08,Караганда,Samsung Galaxy A55


---
### ⭐ Бонус (без автопроверки)

Если у вас есть `orders_raw.csv`, загрузите его в базу как таблицу `orders_raw`:
`pd.read_csv('orders_raw.csv', dtype=str).to_sql('orders_raw', conn, if_exists='replace', index=False)`.

Одним запросом найдите все «подозрительные» строки, где `price` или `qty` пустые, `qty` равно `'-1'` или `status` не входит в список `('оплачен', 'отменён')`.
Какие из ограничений нашей чистой базы (`NOT NULL`, `CHECK`) не пустили бы эти строки?

In [196]:
# ⭐ бонус